# Keypoint-MoSeq Syllable Pipeline

This notebook runs the full [keypoint-moseq](https://keypoint-moseq.readthedocs.io/en/latest/) pipeline on DLC pose data from the **ElevatedMazeFood** experiment. It produces a frame-by-frame syllable time-series that can be aligned with your original videos.

## Prerequisites
- Activate the `kpms` conda environment (see `environment.yml`).
- Place filtered DLC CSV files in `filtered_pose_data/` inside the DLC project directory.

## Pipeline overview
1. Set project paths
2. Setup KPMS project & config
3. Load DLC CSV data
4. Fit PCA
5. Fit AR-HMM model (AR-only → full)
6. Apply model → extract syllable sequences
7. Save syllable time-series CSV
8. (Optional) Export syllable video clips

## 0. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keypoint_moseq as kpms

print("keypoint-moseq version:", kpms.__version__)

## 1. Project paths

Edit `DLC_PROJECT_ROOT` to point to your DLC project folder.

In [ ]:
# ── Edit these paths ────────────────────────────────────────────────────────
DLC_PROJECT_ROOT = os.path.expanduser(
    "~/Downloads/dlc-pose-estimation/ElevatedMazeFood-Atanu-2026-04-04"
)
DLC_CONFIG       = os.path.join(DLC_PROJECT_ROOT, "config.yaml")   # optional
POSE_DIR         = os.path.join(DLC_PROJECT_ROOT, "filtered_pose_data")
VIDEO_DIR        = os.path.join(DLC_PROJECT_ROOT, "videos")
# ── KPMS project dir (will be created if absent) ────────────────────────────
KPMS_PROJECT_DIR = os.path.join(DLC_PROJECT_ROOT, "kpms_project")
# ── Output results dir ──────────────────────────────────────────────────────
RESULTS_DIR      = os.path.join(DLC_PROJECT_ROOT, "results")
# ── Recording parameters ────────────────────────────────────────────────────
FPS              = 30.0   # frames per second
# ────────────────────────────────────────────────────────────────────────────

os.makedirs(KPMS_PROJECT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("DLC project root :", DLC_PROJECT_ROOT)
print("Pose data dir    :", POSE_DIR)
print("KPMS project dir :", KPMS_PROJECT_DIR)
print("Results dir      :", RESULTS_DIR)

## 2. Setup KPMS project and config

In [ ]:
config_path = os.path.join(KPMS_PROJECT_DIR, "config.yml")

if not os.path.exists(config_path):
    if os.path.exists(DLC_CONFIG):
        # Preferred: read bodyparts directly from DLC config.yaml
        kpms.setup_project(KPMS_PROJECT_DIR, deeplabcut_config=DLC_CONFIG)
    else:
        # Fallback: infer bodyparts from first CSV file
        csv_files = sorted([f for f in os.listdir(POSE_DIR) if f.endswith(".csv")])
        sample_csv = os.path.join(POSE_DIR, csv_files[0])
        header = pd.read_csv(sample_csv, header=None, nrows=3)
        bodypart_row = header.iloc[1, 1:].tolist()
        seen, bodyparts = set(), []
        for bp in bodypart_row:
            if bp not in seen:
                seen.add(bp)
                bodyparts.append(bp)
        print("Detected bodyparts:", bodyparts)
        kpms.setup_project(KPMS_PROJECT_DIR, bodyparts=bodyparts)
    print("Created config at", config_path)
else:
    print("Config already exists at", config_path)

config = kpms.load_config(KPMS_PROJECT_DIR)
print("Config loaded.")

### Optional: Tune key config parameters

You can edit `kpms_project/config.yml` directly, or override values here.
Common parameters to adjust:

| Parameter | Default | Notes |
|-----------|---------|-------|
| `num_states` | 100 | Maximum number of syllables |
| `kappa` | auto | Syllable stickiness; higher → fewer, longer syllables |
| `conf_pseudocount` | 0.1 | Low-confidence keypoint masking |
| `anterior_bodyparts` | [] | Front-facing bodyparts for heading angle |
| `posterior_bodyparts` | [] | Rear-facing bodyparts for heading angle |

In [ ]:
# Example overrides – uncomment and adjust as needed
# kpms.update_config(KPMS_PROJECT_DIR,
#     num_states=50,
#     kappa=1e4,
#     anterior_bodyparts=['snout'],
#     posterior_bodyparts=['tailbase'],
# )
# config = kpms.load_config(KPMS_PROJECT_DIR)

## 3. Load DLC CSV data

In [ ]:
print(f"Loading DLC results from: {POSE_DIR}")
coordinates, confidences = kpms.load_deeplabcut_results(POSE_DIR)

print(f"\nLoaded {len(coordinates)} recording(s):")
for name, coords in coordinates.items():
    print(f"  {name}: {coords.shape[0]} frames, {coords.shape[1]} keypoints")

## 4. Fit PCA

In [ ]:
pca_path = os.path.join(KPMS_PROJECT_DIR, "pca.p")

if os.path.exists(pca_path):
    print("Loading existing PCA …")
    pca = kpms.load_pca(KPMS_PROJECT_DIR)
else:
    print("Fitting PCA …")
    formatted = kpms.format_data(coordinates, confidences, **config)
    pca = kpms.fit_pca(**formatted, **config)
    kpms.save_pca(pca, KPMS_PROJECT_DIR)
    print("PCA saved.")

# Show how many PCs explain ≥ 90% variance
kpms.print_dims_to_explain_variance(pca, 0.90)
kpms.plot_scree(pca);

## 5. Format data and initialise model

In [ ]:
data, metadata = kpms.format_data(coordinates, confidences, **config)
model = kpms.init_model(data, pca, **config)
print("Model initialised.")

## 6. Fit model

Two-stage fitting:
1. **AR-only** (fast): learns the autoregressive dynamics without the HMM layer.
2. **Full** (slower): jointly optimises AR + HMM to segment behaviour into syllables.

In [ ]:
NUM_AR_ITERS = 50    # AR-only iterations
NUM_ITERS    = 200   # Full model iterations

print(f"AR-only fitting ({NUM_AR_ITERS} iterations) …")
model, model_name = kpms.fit_model(
    model, data, metadata, KPMS_PROJECT_DIR,
    ar_only=True,
    num_iters=NUM_AR_ITERS,
)

print(f"\nFull model fitting ({NUM_ITERS} iterations) …")
model, model_name = kpms.fit_model(
    model, data, metadata, KPMS_PROJECT_DIR,
    num_iters=NUM_ITERS,
    model_name=model_name,
)

print(f"\nModel saved as: {model_name}")

## 7. Apply model and extract syllable sequences

In [ ]:
print("Applying model to extract syllables …")
results = kpms.apply_model(
    model, pca, data, metadata, KPMS_PROJECT_DIR, **config
)

# Save raw KPMS results (h5 format + per-recording CSVs inside KPMS_PROJECT_DIR)
kpms.save_results_as_csv(results, KPMS_PROJECT_DIR)
print("Raw KPMS results saved.")

## 8. Build syllable time-series CSV

This creates `results/syllable_timeseries.csv` with columns:

| column | description |
|--------|-------------|
| `recording` | filename stem of the recording |
| `frame` | frame index (0-based) |
| `syllable` | integer syllable label |
| `time_sec` | time in seconds at the given FPS |

In [ ]:
rows = []
for recording, rec_results in results.items():
    syllables = np.array(rec_results["syllable"])
    n_frames  = len(syllables)
    times     = np.arange(n_frames) / FPS
    for frame_idx, (syl, t) in enumerate(zip(syllables, times)):
        rows.append({
            "recording": recording,
            "frame":     frame_idx,
            "syllable":  int(syl),
            "time_sec":  round(t, 4),
        })

df = pd.DataFrame(rows, columns=["recording", "frame", "syllable", "time_sec"])

out_csv = os.path.join(RESULTS_DIR, "syllable_timeseries.csv")
df.to_csv(out_csv, index=False)
print(f"Syllable time-series saved to {out_csv}  ({len(df):,} rows)")
df.head(10)

## 9. Visualise syllable usage

In [ ]:
fig, axes = plt.subplots(
    len(results), 1,
    figsize=(14, 2.5 * max(len(results), 1)),
    squeeze=False,
)

for ax, (recording, _) in zip(axes[:, 0], results.items()):
    rec_df = df[df["recording"] == recording]
    ax.scatter(rec_df["time_sec"], rec_df["syllable"],
               s=1, c=rec_df["syllable"], cmap="tab20", alpha=0.7)
    ax.set_title(recording, fontsize=9)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Syllable")

plt.suptitle("Syllable time-series", fontsize=12, y=1.01)
plt.tight_layout()

fig_path = os.path.join(RESULTS_DIR, "syllable_timeseries.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Figure saved to {fig_path}")
plt.show()

## 10. (Optional) Export syllable video clips

This generates short MP4 clips representative of each syllable, placed in `results/syllable_clips/`.
Requires the original video files in `VIDEO_DIR`.

In [ ]:
EXPORT_CLIPS = False   # ← set to True to generate clips

if EXPORT_CLIPS:
    if not os.path.isdir(VIDEO_DIR):
        print(f"Video directory not found: {VIDEO_DIR}")
    else:
        clips_dir = os.path.join(RESULTS_DIR, "syllable_clips")
        os.makedirs(clips_dir, exist_ok=True)
        print(f"Exporting syllable clips to {clips_dir} …")
        kpms.generate_grid_movies(
            results,
            KPMS_PROJECT_DIR,
            video_dir=VIDEO_DIR,
            output_dir=clips_dir,
        )
        print("Done.")
else:
    print("Clip export skipped (set EXPORT_CLIPS = True to enable).")

---
## Summary of outputs

| File | Description |
|------|-------------|
| `kpms_project/config.yml` | KPMS configuration |
| `kpms_project/pca.p` | Fitted PCA |
| `kpms_project/<model_name>/` | Model checkpoints |
| `kpms_project/results.h5` | Raw KPMS results (h5) |
| `results/syllable_timeseries.csv` | Frame-by-frame syllable labels + timestamps |
| `results/syllable_timeseries.png` | Visualisation of syllable sequences |
| `results/syllable_clips/` | (optional) Representative video clips per syllable |